In [1]:
from datasets import load_dataset

dataset = load_dataset("maastrichtlawtech/bsard", 'corpus')

print(dataset)

c:\Users\ghana\OneDrive\kuliah\Semester 8\TA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    corpus: Dataset({
        features: ['id', 'reference', 'article', 'law_type', 'code', 'book', 'part', 'act', 'chapter', 'section', 'subsection', 'description'],
        num_rows: 22633
    })
})


In [6]:
# explore questions — load separately since it's a different config
questions_ds = load_dataset("maastrichtlawtech/bsard", 'questions')
print(questions_ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'category', 'subcategory', 'question', 'extra_description', 'article_ids'],
        num_rows: 886
    })
    synthetic: Dataset({
        features: ['id', 'category', 'subcategory', 'question', 'extra_description', 'article_ids'],
        num_rows: 113165
    })
    test: Dataset({
        features: ['id', 'category', 'subcategory', 'question', 'extra_description', 'article_ids'],
        num_rows: 222
    })
})


In [7]:
# explore corpus schema
corpus = dataset['corpus']
print(f"Corpus size: {len(corpus)}")
print(f"Sample entry keys: {list(corpus[0].keys())}")
print(f"\nSample corpus entry:")
for k, v in corpus[0].items():
    print(f"  {k}: {repr(v)[:120]}")

Corpus size: 22633
Sample entry keys: ['id', 'reference', 'article', 'law_type', 'code', 'book', 'part', 'act', 'chapter', 'section', 'subsection', 'description']

Sample corpus entry:
  id: 1
  reference: "Art. 1.1.1, Code Bruxellois de l'Air, du Climat et de la Maîtrise de l'Energie (Livre 1er, Titre 1er)"
  article: "Le présent Code règle une matière visée à l'article 39 de la Constitution."
  law_type: 'regional'
  code: "Code Bruxellois de l'Air, du Climat et de la Maîtrise de l'Energie"
  book: 'Dispositions communes'
  part: None
  act: 'Généralités'
  chapter: None
  section: None
  subsection: None
  description: 'Dispositions communes, Généralités'


In [8]:
# explore questions schema — splits are separate keys in DatasetDict
train_questions = list(questions_ds['train'])
test_questions = list(questions_ds['test'])

print(f"Train questions: {len(train_questions)}")
print(f"Test questions: {len(test_questions)}")
print(f"Synthetic questions: {len(questions_ds['synthetic'])} (excluded)")
print(f"\nSample entry keys: {list(train_questions[0].keys())}")
print(f"\nSample question entry:")
for k, v in train_questions[0].items():
    print(f"  {k}: {repr(v)[:200]}")

Train questions: 886
Test questions: 222
Synthetic questions: 113165 (excluded)

Sample entry keys: ['id', 'category', 'subcategory', 'question', 'extra_description', 'article_ids']

Sample question entry:
  id: 1102
  category: 'Travail'
  subcategory: 'Travail et parentalité'
  question: 'Je suis travailleur salarié(e). Puis-je refuser de faire des heures supplémentaires ou de travailler de nuit ?'
  extra_description: 'Pendant la grossesse'
  article_ids: '22225,22226,22227,22228,22229,22230,22231,22232,22233,22234'


In [9]:
# check article_ids format (comma-separated string?)
sample_ids = [q['article_ids'] for q in train_questions[:10]]
for i, ids in enumerate(sample_ids):
    print(f"  q[{i}] article_ids type={type(ids).__name__}, value={repr(ids)[:100]}")

  q[0] article_ids type=str, value='22225,22226,22227,22228,22229,22230,22231,22232,22233,22234'
  q[1] article_ids type=str, value='5853,5854,5855'
  q[2] article_ids type=str, value='1096,1097,1098,1108,1109,1110'
  q[3] article_ids type=str, value='12012,12030,12031,12032,12033,12034,12035'
  q[4] article_ids type=str, value='21114,21115,21116,21117,21118,21119,21120,21121,21122,21123,21124'
  q[5] article_ids type=str, value='838'
  q[6] article_ids type=str, value='1676,2334'
  q[7] article_ids type=str, value='1055,1274'
  q[8] article_ids type=str, value='5732'
  q[9] article_ids type=str, value='856,857,611,614'


## Convert to BEIR format

Target files under `../data/bsard/`:
- `corpus.jsonl` — `{"_id": str, "title": str, "text": str}`
- `queries.jsonl` — `{"_id": str, "text": str, "metadata": {...}}`
- `qrels_train.tsv` / `qrels_test.tsv` — `query_id\tdoc_id\tscore`
- `dataset_stats.json`

In [10]:
import json
import os

OUTPUT_DIR = os.path.join("..", "data", "bsard")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- corpus.jsonl ---
corpus_path = os.path.join(OUTPUT_DIR, "corpus.jsonl")
with open(corpus_path, "w", encoding="utf-8") as f:
    for doc in corpus:
        entry = {
            "_id": str(doc["id"]),
            "title": doc["reference"],
            "text": doc["article"],
        }
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Wrote {len(corpus)} documents to {corpus_path}")

Wrote 22633 documents to ..\data\bsard\corpus.jsonl


In [11]:
# --- queries.jsonl + qrels ---
# train_questions and test_questions already loaded from DatasetDict splits

# build corpus id set for validation
corpus_ids = {str(doc["id"]) for doc in corpus}

queries_path = os.path.join(OUTPUT_DIR, "queries.jsonl")
qrels_train_path = os.path.join(OUTPUT_DIR, "qrels_train.tsv")
qrels_test_path = os.path.join(OUTPUT_DIR, "qrels_test.tsv")

all_questions = train_questions + test_questions
n_train_judgments = 0
n_test_judgments = 0
n_missing_docs = 0

with open(queries_path, "w", encoding="utf-8") as fq, \
     open(qrels_train_path, "w", encoding="utf-8") as ft, \
     open(qrels_test_path, "w", encoding="utf-8") as fte:
    
    ft.write("query_id\tdoc_id\tscore\n")
    fte.write("query_id\tdoc_id\tscore\n")
    
    for idx, q in enumerate(all_questions):
        is_train = idx < len(train_questions)
        qid = f"q{q['id']}"
        query_entry = {
            "_id": qid,
            "text": q["question"],
            "metadata": {
                "category": q.get("category", ""),
                "subcategory": q.get("subcategory", ""),
            },
        }
        fq.write(json.dumps(query_entry, ensure_ascii=False) + "\n")
        
        # parse article_ids
        article_ids_raw = q["article_ids"]
        if isinstance(article_ids_raw, str):
            doc_ids = [aid.strip() for aid in article_ids_raw.split(",") if aid.strip()]
        elif isinstance(article_ids_raw, list):
            doc_ids = [str(aid) for aid in article_ids_raw]
        else:
            doc_ids = []
        
        qrels_file = ft if is_train else fte
        for doc_id in doc_ids:
            if doc_id not in corpus_ids:
                n_missing_docs += 1
                continue
            qrels_file.write(f"{qid}\t{doc_id}\t1\n")
            if is_train:
                n_train_judgments += 1
            else:
                n_test_judgments += 1

print(f"Wrote {len(all_questions)} queries to {queries_path}")
print(f"Train judgments: {n_train_judgments}")
print(f"Test judgments: {n_test_judgments}")
print(f"Missing doc references: {n_missing_docs}")

Wrote 1108 queries to ..\data\bsard\queries.jsonl
Train judgments: 5784
Test judgments: 1061
Missing doc references: 0


In [12]:
# --- dataset_stats.json ---
total_judgments = n_train_judgments + n_test_judgments
n_queries = len(all_questions)

stats = {
    "dataset": "bsard",
    "language": "fr",
    "num_documents": len(corpus),
    "num_queries": n_queries,
    "num_train_queries": len(train_questions),
    "num_test_queries": len(test_questions),
    "num_relevance_judgments": total_judgments,
    "num_train_judgments": n_train_judgments,
    "num_test_judgments": n_test_judgments,
    "avg_relevant_docs_per_query": total_judgments / n_queries if n_queries > 0 else 0,
}

stats_path = os.path.join(OUTPUT_DIR, "dataset_stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(json.dumps(stats, indent=2))

{
  "dataset": "bsard",
  "language": "fr",
  "num_documents": 22633,
  "num_queries": 1108,
  "num_train_queries": 886,
  "num_test_queries": 222,
  "num_relevance_judgments": 6845,
  "num_train_judgments": 5784,
  "num_test_judgments": 1061,
  "avg_relevant_docs_per_query": 6.177797833935018
}


In [13]:
# --- sanity check: verify output files ---
print("=== Verification ===")

# check corpus
with open(os.path.join(OUTPUT_DIR, "corpus.jsonl"), encoding="utf-8") as f:
    first_doc = json.loads(f.readline())
    print(f"corpus.jsonl first entry: {first_doc}")

# check queries
with open(os.path.join(OUTPUT_DIR, "queries.jsonl"), encoding="utf-8") as f:
    first_query = json.loads(f.readline())
    print(f"queries.jsonl first entry _id={first_query['_id']}, text={first_query['text'][:80]}...")

# check qrels
for split in ["train", "test"]:
    path = os.path.join(OUTPUT_DIR, f"qrels_{split}.tsv")
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    print(f"qrels_{split}.tsv: {len(lines)-1} judgments (excl. header)")

print(f"\nAll files written to {os.path.abspath(OUTPUT_DIR)}")

=== Verification ===
corpus.jsonl first entry: {'_id': '1', 'title': "Art. 1.1.1, Code Bruxellois de l'Air, du Climat et de la Maîtrise de l'Energie (Livre 1er, Titre 1er)", 'text': "Le présent Code règle une matière visée à l'article 39 de la Constitution."}
queries.jsonl first entry _id=q1102, text=Je suis travailleur salarié(e). Puis-je refuser de faire des heures supplémentai...
qrels_train.tsv: 5784 judgments (excl. header)
qrels_test.tsv: 1061 judgments (excl. header)

All files written to c:\Users\ghana\OneDrive\kuliah\Semester 8\TA\data\bsard
